# MathNet CNN — Math Problem Detection & Understanding
Uses the `ShadenA/MathNet` dataset with a CNN-based classifier to categorize math problems by topic.

## 1. Install Dependencies

In [ ]:
!pip install datasets transformers torch torchvision matplotlib scikit-learn seaborn pandas numpy

## 2. Load the Dataset

In [ ]:
from datasets import load_dataset

ds = load_dataset("ShadenA/MathNet", "all")
print(ds)
print("\nExample entry:")
print(ds["train"][0])

## 3. Explore the Dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df_train = pd.DataFrame(ds["train"])
print("Columns:", df_train.columns.tolist())
print("Shape:", df_train.shape)
df_train.head()

In [ ]:
# Identify the label / topic column — adjust 'label' if the column name differs
label_col = [c for c in df_train.columns if c in ("label", "topic", "category", "subject", "type")]
label_col = label_col[0] if label_col else df_train.columns[-1]
print("Using label column:", label_col)

plt.figure(figsize=(12, 5))
df_train[label_col].value_counts().plot(kind="bar", color="steelblue")
plt.title("Distribution of Math Problem Categories")
plt.xlabel("Category")
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 4. Preprocess — Tokenise Text for 1-D CNN

In [ ]:
import numpy as np
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# ------------------------------------------------------------------
# Identify the text column (problem statement)
# ------------------------------------------------------------------
text_col = [c for c in df_train.columns if c in ("problem", "question", "text", "Problem", "Question")]
text_col = text_col[0] if text_col else df_train.columns[0]
print("Text column:", text_col, "| Label column:", label_col)

# Encode labels to integers
le = LabelEncoder()
y_all = le.fit_transform(df_train[label_col].astype(str))
num_classes = len(le.classes_)
print(f"Classes ({num_classes}):", le.classes_)

# Tokenise the problem text
MAX_WORDS = 20_000
MAX_LEN   = 128

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(df_train[text_col].astype(str))

X_all = pad_sequences(
    tokenizer.texts_to_sequences(df_train[text_col].astype(str)),
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)
print("X shape:", X_all.shape, "| y shape:", y_all.shape)

## 5. Train / Validation Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)
print("Train:", X_train.shape, "| Val:", X_val.shape)

## 6. Build the 1-D CNN Model

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers

EMBED_DIM = 128

def build_cnn(vocab_size, embed_dim, seq_len, num_classes):
    inp = layers.Input(shape=(seq_len,))

    # Embedding with spatial dropout
    x = layers.Embedding(vocab_size, embed_dim)(inp)
    x = layers.SpatialDropout1D(0.2)(x)

    # Parallel conv branches — kernel sizes 2–7 capture short to long n-gram patterns
    branches = []
    for k in (2, 3, 4, 5, 6, 7):
        b = layers.Conv1D(128, k, activation="relu", padding="same",
                          kernel_regularizer=regularizers.l2(1e-4))(x)
        b = layers.BatchNormalization()(b)
        b = layers.Conv1D(64, k, activation="relu", padding="same",
                          kernel_regularizer=regularizers.l2(1e-4))(b)
        b = layers.BatchNormalization()(b)
        b = layers.GlobalMaxPooling1D()(b)
        branches.append(b)

    # Also add a global average pool branch on the raw embedding for softer context
    avg = layers.GlobalAveragePooling1D()(x)
    branches.append(avg)

    merged = layers.Concatenate()(branches)          # 6*64 + embed_dim = 512-dim

    # Deep classification head
    merged = layers.Dense(512, activation="relu",
                          kernel_regularizer=regularizers.l2(1e-4))(merged)
    merged = layers.BatchNormalization()(merged)
    merged = layers.Dropout(0.5)(merged)

    merged = layers.Dense(256, activation="relu",
                          kernel_regularizer=regularizers.l2(1e-4))(merged)
    merged = layers.BatchNormalization()(merged)
    merged = layers.Dropout(0.4)(merged)

    merged = layers.Dense(128, activation="relu",
                          kernel_regularizer=regularizers.l2(1e-4))(merged)
    merged = layers.BatchNormalization()(merged)
    merged = layers.Dropout(0.3)(merged)

    merged = layers.Dense(64, activation="relu")(merged)
    merged = layers.Dropout(0.2)(merged)

    out = layers.Dense(num_classes, activation="softmax")(merged)

    model = models.Model(inp, out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

model = build_cnn(MAX_WORDS, EMBED_DIM, MAX_LEN, num_classes)
model.summary()

## 7. Train

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

callbacks = [
    EarlyStopping(patience=3, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(factor=0.5, patience=2, verbose=1)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=64,
    callbacks=callbacks
)

## 8. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, metric, title in zip(
    axes,
    [("accuracy", "val_accuracy"), ("loss", "val_loss")],
    ["Accuracy", "Loss"]
):
    ax.plot(history.history[metric[0]], label="Train")
    ax.plot(history.history[metric[1]], label="Val")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.legend()

plt.tight_layout()
plt.show()

## 9. Evaluation — Confusion Matrix & Classification Report

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = np.argmax(model.predict(X_val), axis=1)

print(classification_report(y_val, y_pred, target_names=le.classes_))

cm = confusion_matrix(y_val, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 10. Predict on a New Problem

In [ ]:
def predict_topic(problem_text: str) -> dict:
    """Return predicted topic and confidence scores for a math problem."""
    seq = pad_sequences(
        tokenizer.texts_to_sequences([problem_text]),
        maxlen=MAX_LEN, padding="post", truncating="post"
    )
    probs   = model.predict(seq, verbose=0)[0]
    top_idx = np.argsort(probs)[::-1][:3]
    return {
        "predicted_topic": le.classes_[top_idx[0]],
        "confidence":      float(probs[top_idx[0]]),
        "top_3": [
            {"topic": le.classes_[i], "score": float(probs[i])}
            for i in top_idx
        ]
    }

# ── Test cases ─────────────────────────────────────────────────────────────
test_cases = [
    # Calculus
    ("Calculus – Derivative",
     "Find the derivative of f(x) = 3x^2 + 2x - 5"),
    ("Calculus – Chain rule",
     "Differentiate y = sin(x^3 + 2x) with respect to x using the chain rule"),
    ("Calculus – Integral",
     "Evaluate the integral of x^3 * e^x dx using integration by parts"),
    ("Calculus – Definite integral",
     "Compute the definite integral of sqrt(1 - x^2) from -1 to 1"),
    ("Calculus – Limit",
     "Find the limit as x approaches 0 of (sin(x) / x)"),
    ("Calculus – L'Hopital",
     "Use L'Hopital's rule to evaluate lim x->inf of x * e^(-x)"),
    ("Calculus – Series",
     "Determine whether the series sum_{n=1}^{inf} 1/n^2 converges using the p-test"),
    ("Calculus – Taylor series",
     "Find the Taylor series expansion of e^x centered at x = 0 up to the 4th term"),
    ("Calculus – Partial derivative",
     "Compute the partial derivative of f(x,y) = x^2*y + y^3 with respect to y"),
    ("Calculus – Multivariable",
     "Find the gradient vector of f(x,y,z) = xyz + x^2 + z^3"),

    # Algebra
    ("Algebra – Quadratic",
     "Solve for x: 2x^2 - 4x - 6 = 0 using the quadratic formula"),
    ("Algebra – System of equations",
     "Solve the system: 3x + 2y = 12 and x - y = 1"),
    ("Algebra – Polynomial factoring",
     "Factor completely: x^3 - 6x^2 + 11x - 6"),
    ("Algebra – Logarithm",
     "Solve for x: log_2(x + 3) + log_2(x - 1) = 5"),
    ("Algebra – Exponential",
     "Solve 5^(2x-1) = 125 for x"),
    ("Algebra – Rational expression",
     "Simplify (x^2 - 9) / (x^2 - x - 6)"),
    ("Algebra – Absolute value",
     "Solve |3x - 7| = 14 for all real x"),
    ("Algebra – Inequality",
     "Find all x satisfying x^2 - 5x + 6 < 0"),

    # Geometry
    ("Geometry – Triangle area",
     "Find the area of a triangle with base 8 cm and height 5 cm"),
    ("Geometry – Circle",
     "Calculate the circumference and area of a circle with radius 7"),
    ("Geometry – Volume sphere",
     "What is the volume of a sphere with radius 4?"),
    ("Geometry – Pythagorean theorem",
     "A right triangle has legs of length 6 and 8. Find the hypotenuse."),
    ("Geometry – Surface area cylinder",
     "Find the total surface area of a cylinder with radius 3 and height 10"),
    ("Geometry – Similar triangles",
     "Two similar triangles have corresponding sides 5 and 12. If the perimeter of the smaller is 24, find the perimeter of the larger."),

    # Trigonometry
    ("Trigonometry – Identity",
     "Prove that sin^2(x) + cos^2(x) = 1"),
    ("Trigonometry – Angle",
     "Find all angles theta in [0, 2pi) satisfying 2*cos(theta) - 1 = 0"),
    ("Trigonometry – Law of cosines",
     "In triangle ABC, a=7, b=9, C=60 degrees. Find side c using the law of cosines."),
    ("Trigonometry – Inverse trig",
     "Evaluate arctan(sqrt(3)) and express the answer in radians"),

    # Statistics & Probability
    ("Probability – Basic",
     "A bag has 4 red and 6 blue marbles. What is the probability of drawing 2 red marbles without replacement?"),
    ("Probability – Binomial",
     "A fair coin is flipped 10 times. What is the probability of getting exactly 6 heads?"),
    ("Statistics – Mean/Variance",
     "Find the mean and variance of the dataset: 4, 8, 15, 16, 23, 42"),
    ("Statistics – Normal distribution",
     "Given X ~ N(50, 25), find P(X > 60) using the standard normal table"),
    ("Statistics – Hypothesis test",
     "Perform a one-sample t-test for mu=100 given sample mean 98, s=5, n=25 at alpha=0.05"),
    ("Statistics – Regression",
     "Find the least-squares regression line for the data points (1,2),(2,4),(3,5),(4,4),(5,5)"),

    # Linear Algebra
    ("Linear Algebra – Matrix multiplication",
     "Compute the product of A = [[1,2],[3,4]] and B = [[5,6],[7,8]]"),
    ("Linear Algebra – Determinant",
     "Find the determinant of the 3x3 matrix [[1,2,3],[4,5,6],[7,8,9]]"),
    ("Linear Algebra – Eigenvalues",
     "Find the eigenvalues and eigenvectors of A = [[4,1],[2,3]]"),
    ("Linear Algebra – Inverse",
     "Compute the inverse of the matrix [[2,1],[5,3]]"),
    ("Linear Algebra – Span",
     "Determine whether the vectors [1,2,3], [4,5,6], [7,8,9] are linearly independent"),

    # Number Theory
    ("Number Theory – GCD",
     "Find the greatest common divisor of 252 and 198 using the Euclidean algorithm"),
    ("Number Theory – Prime",
     "Prove that there are infinitely many prime numbers"),
    ("Number Theory – Modular arithmetic",
     "Find 3^100 mod 7 using Fermat's little theorem"),
    ("Number Theory – Diophantine",
     "Find all integer solutions to 15x + 21y = 3"),

    # Differential Equations
    ("ODE – First order",
     "Solve the ODE dy/dx = 3y with initial condition y(0) = 2"),
    ("ODE – Separable",
     "Solve dy/dx = (x^2) / (1 - y^2) by separating variables"),
    ("ODE – Second order",
     "Find the general solution of y'' - 5y' + 6y = 0"),
    ("ODE – Homogeneous",
     "Solve the initial value problem y'' + 4y = 0, y(0)=1, y'(0)=0"),

    # Complex Numbers
    ("Complex – Arithmetic",
     "Compute (3 + 4i)^2 and express in the form a + bi"),
    ("Complex – Modulus",
     "Find the modulus and argument of the complex number -1 + i*sqrt(3)"),
    ("Complex – De Moivre",
     "Use De Moivre's theorem to find (cos(pi/6) + i*sin(pi/6))^12"),

    # Combinatorics
    ("Combinatorics – Permutations",
     "How many ways can 8 people be seated in a row of 8 chairs?"),
    ("Combinatorics – Combinations",
     "In how many ways can a committee of 4 be chosen from 10 people?"),
    ("Combinatorics – Pigeonhole",
     "If 13 socks of 4 colors are in a drawer, how many must you draw to guarantee a matching pair?"),

    # Set Theory / Logic
    ("Set Theory – Venn",
     "Given |A|=30, |B|=25, |A∩B|=10, find |A∪B|"),
    ("Logic – Proof by induction",
     "Prove by induction that the sum of the first n positive integers equals n(n+1)/2"),
]

print(f"{'#':<4} {'Category':<35} {'Predicted':<30} {'Conf':>6}")
print("-" * 82)
for i, (category, problem) in enumerate(test_cases, 1):
    result = predict_topic(problem)
    predicted = result["predicted_topic"]
    conf      = result["confidence"]
    marker    = "" if conf >= 0.5 else " (?)"
    print(f"{i:<4} {category:<35} {predicted:<30} {conf:>5.1%}{marker}")

print("\n--- Top-3 detail for first problem in each math area ---")
area_firsts = {
    "Calculus":     test_cases[0],
    "Algebra":      test_cases[10],
    "Geometry":     test_cases[18],
    "Trigonometry": test_cases[24],
    "Probability":  test_cases[28],
    "Linear Alg":   test_cases[34],
    "Number Theory":test_cases[38],
    "ODE":          test_cases[42],
    "Complex":      test_cases[46],
    "Combinatorics":test_cases[49],
}
for area, (cat, prob) in area_firsts.items():
    r = predict_topic(prob)
    print(f"\n[{area}] {prob[:70]}")
    for t in r["top_3"]:
        print(f"  {t['topic']:<30} {t['score']:.2%}")

## 11. Save the Model

In [ ]:
import pickle, os

os.makedirs("saved_model", exist_ok=True)
model.save("saved_model/mathnet_cnn.keras")

with open("saved_model/tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)
with open("saved_model/label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

print("Model, tokenizer, and label encoder saved to saved_model/")